# nb01 - Clean: CMS Star Ratings

**Run locally after nb00.** All raw-to-clean transformation for this project lives in this notebook, for reproducibility.

Two outputs:
1. `../data/star_summary_clean.csv` - one row per contract per year (Overall / Part C / Part D stars)
2. `../data/star_measures_clean.csv` - one row per contract per year **per measure**, with its Domain

**File quirks handled here:**
- The CMS CSVs are Windows-1252 encoded, not UTF-8 (smart apostrophes in measure names).
- Every file has a **title row** on top.
- The Measure Stars file has a **two-row header**: the upper row holds the Domain names (merged across columns, so they read as blanks), and the row beneath it holds the actual measure names.
- Beneath that header sits a **measurement-period row** (values like `01/01/2024 - 12/31/2024`) with no contract id. If it is left in, `star_to_num` pulls the leading `0` out of the date and invents a contract with 0.0 stars, one per measure per year (137 rows), which drags every national benchmark down. It is dropped here.
- CMS writes the same parent company under **more than one spelling** (`Molina Healthcare, Inc.` and `Molina Healthcare, Inc.,`; `Samaritan Health Services` and `Samaritan Health Services, Inc.`), and truncates the field at 50 characters. `ParentOrgGroup` merges the duplicates so the notebook matches the `Parent Org (group)` field in Tableau. **Merging changes numbers; relabelling does not.**

In [1]:
import json
from pathlib import Path
import pandas as pd

DATA = Path('..') / 'data'
RAW = DATA / 'raw'
years = [f['year'] for f in json.loads((DATA / 'extraction_log.json').read_text())['files']]
print('years:', years)

def find_csv(yr, needle):
    hits = [p for p in (RAW / yr).rglob('*.csv') if needle in p.name]
    assert len(hits) == 1, (yr, needle, [h.name for h in hits])
    return hits[0]

def read_cms_csv(path, **kw):
    # CMS files are Windows-1252, not UTF-8
    for enc in ('utf-8', 'cp1252', 'latin-1'):
        try:
            return pd.read_csv(path, encoding=enc, **kw)
        except UnicodeDecodeError:
            continue
    raise UnicodeDecodeError(f'could not decode {path}')

def star_to_num(series):
    # leading star number (1 to 5, half steps); text like 'Not enough data available' -> NaN
    return pd.to_numeric(series.astype(str).str.extract(r'(\d(?:\.\d)?)')[0], errors='coerce')

# --- Parent organization cleanup -------------------------------------------------
# MERGES change numbers: CMS spells one company two ways, so a naive groupby splits it
# in two. These are the same merges made in Tableau's `Parent Org (group)` field.
PARENT_ORG_MERGES = {
    'Molina Healthcare, Inc.,':        'Molina Healthcare, Inc.',
    'Samaritan Health Services':       'Samaritan Health Services, Inc.',
}

# RENAMES are display only and never change a number. CMS shouts 12 names in ALL CAPS.
PARENT_ORG_RENAMES = {
    'CAPITAL BLUE CROSS':                                 'Capital Blue Cross',
    'DOCTORS HEALTHCARE PLANS, INC.':                     'Doctors HealthCare Plans, Inc.',
    'IMPERIAL COUNTY LOCAL HEALTH AUTHORITY':             'Imperial County Local Health Authority',
    'INLAND EMPIRE HEALTH PLAN':                          'Inland Empire Health Plan',
    'MEDICAL MUTUAL OF OHIO':                             'Medical Mutual of Ohio',
    'NEIGHBORHOOD HEALTH PLAN OF RHODE ISLAND':           'Neighborhood Health Plan of Rhode Island',
    'SAN FRANCISCO HEALTH AUTHORITY':                     'San Francisco Health Authority',
    'SAN JOAQUIN COUNTY HEALTH COMMISSION':               'San Joaquin County Health Commission',
    'SANTA BARBARA SAN LUIS OBISPO REGIONAL HEALTH AUTH': 'Santa Barbara San Luis Obispo Regional Health Auth',
    'SANTA CLARA COUNTY HEALTH AUTHORITY':                'Santa Clara County Health Authority',
    'SANTA CRUZ MONTEREY MERCED SAN BENITO MARIPOSA MAN': 'Santa Cruz Monterey Merced San Benito Mariposa Man',
    'VISITING NURSE ASSOCIATION OF CENTRAL NEW YORK':     'Visiting Nurse Association of Central New York',
}

def parent_org_group(series):
    """Reproduce Tableau's `Parent Org (group)` field: merge the duplicate spellings first,
    then apply the display renames."""
    return series.replace(PARENT_ORG_MERGES).replace(PARENT_ORG_RENAMES)


years: ['2024', '2025', '2026']


## Part 1 - Summary ratings (one row per contract per year)

In [2]:
frames = []
for yr in years:
    f = find_csv(yr, 'Summary Rating')
    df = read_cms_csv(f, skiprows=1, dtype=str)   # skip the title row
    df.columns = [str(c).strip() for c in df.columns]

    partc   = [c for c in df.columns if 'Part C Summary' in c][0]
    partd   = [c for c in df.columns if 'Part D Summary' in c][0]
    overall = [c for c in df.columns if c.strip().endswith('Overall')][0]

    out = pd.DataFrame({
        'Year': int(yr),
        'Contract':      df['Contract Number'].str.strip(),
        'OrgType':       df['Organization Type'].str.strip(),
        'ContractName':  df['Contract Name'].str.strip(),
        'MarketingName': df['Organization Marketing Name'].str.strip(),
        'ParentOrg':     df['Parent Organization'].str.strip(),
        'SNP':           df['SNP'].str.strip() if 'SNP' in df.columns else pd.NA,
        'PartC_Stars':   star_to_num(df[partc]),
        'PartD_Stars':   star_to_num(df[partd]),
        'Overall_Stars': star_to_num(df[overall]),
    })
    print(f'{yr}: {len(out)} rows | overall rated: {out.Overall_Stars.notna().sum()}')
    frames.append(out)

stars = pd.concat(frames, ignore_index=True)

# reproduce Tableau's `Parent Org (group)` so parent-org figures match the workbook
stars['ParentOrgGroup'] = parent_org_group(stars['ParentOrg'])
print('\nparent orgs: %d raw -> %d after merging duplicate spellings'
      % (stars.ParentOrg.nunique(), stars.ParentOrgGroup.nunique()))

stars.to_csv(DATA / 'star_summary_clean.csv', index=False)
print('\nwrote star_summary_clean.csv', stars.shape)
print('\nnational average OVERALL star per year (the benchmark):')
print(stars.groupby('Year').Overall_Stars.mean().round(3))

2024: 857 rows | overall rated: 545
2025: 789 rows | overall rated: 521
2026: 769 rows | overall rated: 516

parent orgs: 219 raw -> 217 after merging duplicate spellings

wrote star_summary_clean.csv (2415, 11)

national average OVERALL star per year (the benchmark):
Year
2024    3.682
2025    3.650
2026    3.652
Name: Overall_Stars, dtype: float64


## Part 2 - Measure-level stars (two-row header handled)

Read with no header, then rebuild the column names: the ID columns take their names from the upper (Domain) row, and every measure column takes its name from the lower (Measure) row. The Domain row is forward filled so each measure keeps the domain it belongs to. Then unpivot wide to long.

In [3]:
mframes = []
for yr in years:
    f = find_csv(yr, 'Measure Stars')
    raw = read_cms_csv(f, skiprows=1, header=None, dtype=str)   # skip title; keep both header rows as data

    hdr_domain  = raw.iloc[0]   # upper header row: ID names + Domain names (merged -> blanks)
    hdr_measure = raw.iloc[1]   # lower header row: blanks over the IDs, then measure names

    # the leading columns where the MEASURE row is blank are the ID columns
    def _blank(v):
        return pd.isna(v) or str(v).strip() == ''
    n_id = 0
    while n_id < len(hdr_measure) and _blank(hdr_measure.iloc[n_id]):
        n_id += 1
    if n_id == 0:
        n_id = 5   # fallback

    id_names      = [str(hdr_domain.iloc[i]).strip() for i in range(n_id)]
    measure_names = [str(hdr_measure.iloc[i]).strip() for i in range(n_id, raw.shape[1])]
    # each measure column inherits the domain above it (forward fill across the merged cells)
    domains = (hdr_domain.iloc[n_id:].ffill().fillna('').astype(str).str.strip().tolist())
    measure_to_domain = dict(zip(measure_names, domains))

    data = raw.iloc[2:].reset_index(drop=True)
    data.columns = id_names + measure_names

    # strip whitespace on the ID values (contract IDs carried trailing spaces)
    for c in id_names:
        data[c] = data[c].astype(str).str.strip().replace({'nan': pd.NA, '': pd.NA})

    print(f'\n{yr}: {n_id} ID columns, {len(measure_names)} measure columns')
    for m in measure_names[:5]:
        print(f'     {m}   [domain: {measure_to_domain[m][:40]}]')
    print(f'     ... and {len(measure_names)-5} more')

    id_col = 'CONTRACT_ID' if 'CONTRACT_ID' in id_names else 'Contract Number'

    # CMS writes a measurement-period row under the header ('01/01/2024 - 12/31/2024').
    # It carries no contract id. Left in, star_to_num pulls the leading '0' out of that
    # date and invents a contract scoring 0.0 stars on every measure, which drags every
    # national benchmark down. Drop any row without a contract id.
    n_before = len(data)
    data = data[data[id_col].notna()].reset_index(drop=True)
    print(f'     dropped {n_before - len(data)} row(s) with no contract id (measurement-period row)')

    long = data.melt(id_vars=id_names, value_vars=measure_names,
                     var_name='MeasureLabel', value_name='StarsRaw')
    long = long.rename(columns={id_col: 'Contract',
                                'Organization Type': 'OrgType',
                                'Organization Marketing Name': 'MarketingName',
                                'Parent Organization': 'ParentOrg'})
    long['Year'] = int(yr)
    long['Domain'] = long['MeasureLabel'].map(measure_to_domain)

    sp = long['MeasureLabel'].str.split(':', n=1, expand=True)
    long['MeasureCode'] = sp[0].str.strip()
    long['MeasureName'] = sp[1].str.strip() if sp.shape[1] > 1 else long['MeasureLabel']
    long['Stars'] = star_to_num(long['StarsRaw'])

    keep = ['Year','Contract','OrgType','MarketingName','ParentOrg',
            'Domain','MeasureCode','MeasureName','StarsRaw','Stars']
    mframes.append(long[[c for c in keep if c in long.columns]])

measures = pd.concat(mframes, ignore_index=True)

# same parent-org grouping as the summary file
measures['ParentOrgGroup'] = parent_org_group(measures['ParentOrg'])

assert measures.Contract.isna().sum() == 0, 'rows with no contract id survived the filter'
assert (measures.Stars.dropna() >= 1).all(), 'a 0.0-star row survived; the period row is back'

measures.to_csv(DATA / 'star_measures_clean.csv', index=False)
print('\nwrote star_measures_clean.csv', measures.shape)
print('rated measure rows:', measures.Stars.notna().sum())
print('sample contracts:', list(measures.Contract.dropna().unique()[:5]))
print('H1224 rows found:', int((measures.Contract == 'H1224').sum()))


2024: 5 ID columns, 46 measure columns
     C01: Breast Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C02: Colorectal Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C03: Annual Flu Vaccine   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C04: Monitoring Physical Activity   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C05: Special Needs Plan (SNP) Care Management   [domain: HD2: Managing Chronic (Long Term) Condit]
     ... and 41 more
     dropped 1 row(s) with no contract id (measurement-period row)



2025: 5 ID columns, 46 measure columns
     C01: Breast Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C02: Colorectal Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C03: Annual Flu Vaccine   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C04: Monitoring Physical Activity   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C05: Special Needs Plan (SNP) Care Management   [domain: HD2: Managing Chronic (Long Term) Condit]
     ... and 41 more
     dropped 1 row(s) with no contract id (measurement-period row)



2026: 5 ID columns, 45 measure columns
     C01: Breast Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C02: Colorectal Cancer Screening   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C03: Annual Flu Vaccine   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C04: Improving or Maintaining Physical Health   [domain: HD1: Staying Healthy: Screenings, Tests ]
     C05: Improving or Maintaining Mental Health   [domain: HD1: Staying Healthy: Screenings, Tests ]
     ... and 40 more
     dropped 1 row(s) with no contract id (measurement-period row)



wrote star_measures_clean.csv (110321, 11)
rated measure rows: 66206
sample contracts: ['E3014', 'H0022', 'H0028', 'H0029', 'H0034']
H1224 rows found: 137


## QA - where does L.A. Care (H1224) lose stars?

In [4]:
LA = 'H1224'

print('=== L.A. Care overall stars by year ===')
print(stars[stars.Contract == LA][['Year','MarketingName','Overall_Stars','PartC_Stars','PartD_Stars']].to_string(index=False))

yr_latest = max(int(y) for y in years)
la_rows = measures[(measures.Contract == LA) & (measures.Year == yr_latest)]
print(f'\nL.A. Care rows in {yr_latest}: {len(la_rows)} | rated: {la_rows.Stars.notna().sum()}')
print('sample raw values:', list(la_rows.StarsRaw.dropna().unique()[:6]))

nat = (measures[measures.Stars.notna()]
       .groupby(['Year','MeasureCode','MeasureName'])['Stars'].mean().round(2).rename('NationalAvg'))
la  = (la_rows.set_index(['Year','MeasureCode','MeasureName'])['Stars'].rename('LACare'))

cmp = pd.concat([la, nat], axis=1, join='inner').dropna(subset=['LACare'])
cmp['Gap'] = (cmp['LACare'] - cmp['NationalAvg']).round(2)

print(f'\n=== {yr_latest}: measures where L.A. Care is FURTHEST BELOW the national average ===')
print(cmp.sort_values('Gap').head(12).to_string())

print(f'\n=== {yr_latest}: measures where L.A. Care is ABOVE the national average ===')
print(cmp.sort_values('Gap', ascending=False).head(8).to_string())

print('\n=== L.A. Care average stars by domain ===')
print(la_rows[la_rows.Stars.notna()].groupby('Domain').Stars.agg(['mean','count']).round(2).to_string())

=== L.A. Care overall stars by year ===
 Year         MarketingName  Overall_Stars  PartC_Stars  PartD_Stars
 2024 L.A. Care Health Plan            NaN          NaN          NaN
 2025 L.A. Care Health Plan            3.0          2.5          3.5
 2026 L.A. Care Health Plan            3.0          3.0          4.0

L.A. Care rows in 2026: 45 | rated: 43
sample raw values: ['2', '3', '4', 'Plan too new to be measured ', '5', '1']

=== 2026: measures where L.A. Care is FURTHEST BELOW the national average ===
                                                            LACare  NationalAvg   Gap
Year MeasureCode MeasureName                                                         
2026 C17         Medication Reconciliation Post-Discharge      1.0         3.83 -2.83
     C27         Care Coordination                             1.0         3.50 -2.50
     C24         Customer Service                              1.0         3.47 -2.47
     C08         Care for Older Adults – Medication Review

## Part 3 - Measure weights and data sources (a reference table CMS does not ship)

The Star Ratings data tables contain **no weights**. But CMS does not average the measures, it weights them from 1 to 5, and the weights vary by a factor of five. Without them, a measure gap chart silently treats a triple weighted readmissions measure as equal to a single weighted falls measure.

The weights are published separately, in a PDF: [2026 Star Ratings Measures and Weights](https://www.cms.gov/files/document/2026-star-ratings-measures.pdf). The data source for each measure (HEDIS, CAHPS, HOS, pharmacy claims, CMS administrative data) comes from the [2026 Technical Notes](https://www.cms.gov/files/document/2026-star-ratings-technical-notes.pdf).

This cell hand keys both into a reference table so they can be joined onto the measure data.

**Scoped to 2026 on purpose.** Weights change between rating years (Patients' Experience measures dropped from a weight of 4 to a weight of 2 for 2026). The table is keyed on **Year and Measure Code**, so 2024 and 2025 rows come back with a null weight rather than silently inheriting the wrong one.

In [5]:
# CMS 2026 Star Ratings, per measure: weight, weighting category, and where the number comes from.
# Keyed on the 2026 measure codes. Sources are in the markdown above.
#
# weight 1 = Process        2 = Patients' Experience / Complaints / Access
#        3 = Outcome        5 = Improvement

WEIGHTS_2026 = [
    # code, weight, weighting category,        data source
    ('C01', 1, 'Process',              'HEDIS (claims and records)'),
    ('C02', 1, 'Process',              'HEDIS (claims and records)'),
    ('C03', 1, 'Process',              'CAHPS survey'),
    ('C04', 1, 'Outcome',              'HOS survey'),
    ('C05', 1, 'Outcome',              'HOS survey'),
    ('C06', 1, 'Process',              'HOS survey'),
    ('C07', 1, 'Process',              'HEDIS (claims and records)'),
    ('C08', 1, 'Process',              'HEDIS (claims and records)'),
    ('C09', 1, 'Process',              'HEDIS (claims and records)'),
    ('C10', 1, 'Process',              'HEDIS (claims and records)'),
    ('C11', 1, 'Process',              'HEDIS (claims and records)'),
    ('C12', 3, 'Intermediate Outcome', 'HEDIS (claims and records)'),
    ('C13', 1, 'Process',              'HEDIS (claims and records)'),
    ('C14', 3, 'Intermediate Outcome', 'HEDIS (claims and records)'),
    ('C15', 1, 'Process',              'HOS survey'),
    ('C16', 1, 'Process',              'HOS survey'),
    ('C17', 1, 'Process',              'HEDIS (claims and records)'),
    ('C18', 3, 'Outcome',              'HEDIS (claims and records)'),
    ('C19', 1, 'Process',              'HEDIS (claims and records)'),
    ('C20', 1, 'Process',              'HEDIS (claims and records)'),
    ('C21', 1, 'Process',              'HEDIS (claims and records)'),
    ('C22', 2, "Patients' Experience", 'CAHPS survey'),
    ('C23', 2, "Patients' Experience", 'CAHPS survey'),
    ('C24', 2, "Patients' Experience", 'CAHPS survey'),
    ('C25', 2, "Patients' Experience", 'CAHPS survey'),
    ('C26', 2, "Patients' Experience", 'CAHPS survey'),
    ('C27', 2, "Patients' Experience", 'CAHPS survey'),
    ('C28', 2, "Patients' Experience", 'CMS complaints database'),
    ('C29', 2, "Patients' Experience", 'CMS enrollment records'),
    ('C30', 5, 'Improvement',          "Computed from the plan's own prior year"),
    ('C31', 2, 'Access',               'CMS independent review entity'),
    ('C32', 2, 'Access',               'CMS independent review entity'),
    ('C33', 2, 'Access',               'CMS secret shopper calls'),
    ('D01', 2, 'Access',               'CMS secret shopper calls'),
    ('D02', 2, "Patients' Experience", 'CMS complaints database'),
    ('D03', 2, "Patients' Experience", 'CMS enrollment records'),
    ('D04', 5, 'Improvement',          "Computed from the plan's own prior year"),
    ('D05', 2, "Patients' Experience", 'CAHPS survey'),
    ('D06', 2, "Patients' Experience", 'CAHPS survey'),
    ('D07', 1, 'Process',              'Plan submitted pricing files'),
    ('D08', 3, 'Intermediate Outcome', 'Pharmacy claims (PQA)'),
    ('D09', 3, 'Intermediate Outcome', 'Pharmacy claims (PQA)'),
    ('D10', 3, 'Intermediate Outcome', 'Pharmacy claims (PQA)'),
    ('D11', 1, 'Process',              'Plan reported to CMS'),
    ('D12', 1, 'Process',              'Pharmacy claims (PQA)'),
]

weights = pd.DataFrame(WEIGHTS_2026, columns=['MeasureCode', 'Weight', 'WeightCategory', 'DataSource'])
weights.insert(0, 'Year', 2026)
weights['Part'] = weights.MeasureCode.str[0].map({'C': 'Part C (health plan)', 'D': 'Part D (drug plan)'})

# attach the measure name straight from the data, so the labels cannot drift apart
names = (measures[(measures.Year == 2026) & measures.MeasureCode.notna()][['MeasureCode', 'MeasureName']]
         .drop_duplicates())
weights = weights.merge(names, on='MeasureCode', how='left')

# --- guardrails: this table is hand keyed, so it gets checked ---
codes_in_data = set(measures[(measures.Year == 2026) & measures.MeasureCode.notna()].MeasureCode)
assert set(weights.MeasureCode) == codes_in_data, (
    'weights table and 2026 data disagree on the measure list: '
    f'{codes_in_data ^ set(weights.MeasureCode)}')
assert weights.MeasureName.notna().all(), 'a weight row failed to pick up a measure name'
assert weights.Weight.isin([1, 2, 3, 5]).all(), 'unexpected weight value'

weights = weights[['Year', 'MeasureCode', 'MeasureName', 'Part', 'Weight', 'WeightCategory', 'DataSource']]
weights.to_csv(DATA / 'measure_weights.csv', index=False)

print('wrote measure_weights.csv', weights.shape)
print('\nmeasures by weight:')
print(weights.groupby('Weight').size().rename('measures').to_string())
print('\nmeasures by data source:')
print(weights.groupby('DataSource').size().rename('measures').sort_values(ascending=False).to_string())

wrote measure_weights.csv (45, 7)

measures by weight:
Weight
1    21
2    16
3     6
5     2

measures by data source:
DataSource
HEDIS (claims and records)                 15
CAHPS survey                                9
HOS survey                                  5
Pharmacy claims (PQA)                       4
CMS complaints database                     2
CMS enrollment records                      2
CMS independent review entity               2
Computed from the plan's own prior year     2
CMS secret shopper calls                    2
Plan reported to CMS                        1
Plan submitted pricing files                1


## QA - can the assigned star be rebuilt from the measures?

If the hand keyed weights are right, it should be possible to start from the individual measure stars and land back on the summary star CMS assigned. Doing so is the strongest available check on the weights table.

CMS builds a summary rating in four steps:

1. **Weighted mean** of the measure stars, using the weights above
2. **Reward factor**, a bonus for contracts with consistently high and low variance performance (not published in the data tables)
3. **Categorical Adjustment Index (CAI)**, an adjustment for the share of the contract's members who are low income subsidy or dual eligible (LIS/DE) and the share who are disabled
4. **Round** to the nearest half star

Step 3 is the interesting one and it is why the raw archive ships a CAI file. That file does **not** give the adjustment value, only a *Final Adjustment Category* per contract. The values themselves live in a separate PDF, the [2026 Categorical Adjustment Index Measure Supplement](https://www.cms.gov/files/document/2026-categorical-adjustment-index-measure-supplement.pdf), and are keyed in below.

Step 2 cannot be reproduced, because CMS does not publish the reward factor per contract. So an exact match is not expected for every contract, and that gap is itself the finding.

In [6]:
import numpy as np

# 2026 CAI values, by Final Adjustment Category.
# Source: CMS 2026 Categorical Adjustment Index Measure Supplement, Tables 7, 10, and 13.
CAI_OVERALL = {1:-0.063262, 2:-0.040422, 3:-0.017803, 4:0.003256, 5:0.018790,
               6: 0.045683, 7: 0.058145, 8: 0.101257, 9:0.145515}
CAI_PART_C  = {1:-0.058259, 2:-0.036927, 3:-0.013699, 4:0.004022, 5:0.032302,
               6: 0.059788, 7: 0.080451, 8: 0.102370}
CAI_PART_D  = {1:-0.033144, 2:-0.014987, 3:-0.002688, 4:0.046282, 5:0.072332, 6:0.128476}

# each contract's Final Adjustment Category comes from the CMS CAI data table
cai_raw = read_cms_csv(find_csv('2026', 'CAI'), skiprows=1, dtype=str)
cai_raw.columns = [c.strip() for c in cai_raw.columns]
cai = pd.DataFrame({
    'Contract':     cai_raw['Contract Number'].str.strip(),
    'PartC_FAC':    pd.to_numeric(cai_raw['Part C FAC'].str.strip(),        errors='coerce'),
    'PartD_FAC':    pd.to_numeric(cai_raw['Part D MA-PD FAC'].str.strip(),  errors='coerce'),
    'Overall_FAC':  pd.to_numeric(cai_raw['Overall FAC'].str.strip(),       errors='coerce'),
})
cai['PartC_CAI']   = cai.PartC_FAC.map(CAI_PART_C)
cai['PartD_CAI']   = cai.PartD_FAC.map(CAI_PART_D)
cai['Overall_CAI'] = cai.Overall_FAC.map(CAI_OVERALL)
cai.to_csv(DATA / 'contract_cai.csv', index=False)
print('wrote contract_cai.csv', cai.shape)

half_star = lambda x: np.round(x * 2) / 2          # CMS rounds summary ratings to the nearest half star

scored = (measures[(measures.Year == 2026) & measures.Stars.notna()]
          .merge(weights[['MeasureCode', 'Weight']], on='MeasureCode'))
scored['wx'] = scored.Stars * scored.Weight

def rebuild(df, fac_col, cai_col, assigned_col, label):
    g = (df.groupby('Contract')
           .agg(measures=('Stars', 'size'), wx=('wx', 'sum'), wt=('Weight', 'sum')))
    g['weighted_mean'] = g.wx / g.wt
    g = (g.join(cai.set_index('Contract')[[fac_col, cai_col]])
           .join(stars[stars.Year == 2026].set_index('Contract')[assigned_col])
           .dropna(subset=[cai_col, assigned_col]))
    g['rebuilt']         = half_star(g.weighted_mean + g[cai_col])
    g['rebuilt_no_cai']  = half_star(g.weighted_mean)

    exact  = (g.rebuilt == g[assigned_col]).mean()
    close  = ((g.rebuilt - g[assigned_col]).abs() <= 0.5).mean()
    no_cai = (g.rebuilt_no_cai == g[assigned_col]).mean()
    moved  = (g.rebuilt != g.rebuilt_no_cai).sum()
    print(f'{label:8s} {len(g):4d} contracts | exact {exact:5.1%} | within half a star {close:6.1%} '
          f'| without CAI {no_cai:5.1%} | CAI moves the star for {moved} contracts ({moved/len(g):.0%})')
    return g

print('\nRebuilding the assigned star from the measure stars:\n')
pc = rebuild(scored[scored.MeasureCode.str.startswith('C')], 'PartC_FAC', 'PartC_CAI', 'PartC_Stars',   'Part C')
pdd = rebuild(scored[scored.MeasureCode.str.startswith('D')], 'PartD_FAC', 'PartD_CAI', 'PartD_Stars',   'Part D')
ov = rebuild(scored,                                          'Overall_FAC','Overall_CAI','Overall_Stars','Overall')

# the weights cannot be badly wrong if every contract lands within half a star
assert ((pc.rebuilt - pc.PartC_Stars).abs() <= 0.5).all(), 'a Part C rebuild is off by more than half a star'

print('\n=== L.A. Care (H1224), 2026 ===')
la = pd.DataFrame({
    'weighted mean of measures': [pc.loc[LA,'weighted_mean'], pdd.loc[LA,'weighted_mean'], ov.loc[LA,'weighted_mean']],
    'Final Adjustment Category': [pc.loc[LA,'PartC_FAC'],     pdd.loc[LA,'PartD_FAC'],     ov.loc[LA,'Overall_FAC']],
    'CAI added':                 [pc.loc[LA,'PartC_CAI'],     pdd.loc[LA,'PartD_CAI'],     ov.loc[LA,'Overall_CAI']],
    'rebuilt star':              [pc.loc[LA,'rebuilt'],       pdd.loc[LA,'rebuilt'],       ov.loc[LA,'rebuilt']],
    'CMS assigned':              [pc.loc[LA,'PartC_Stars'],   pdd.loc[LA,'PartD_Stars'],   ov.loc[LA,'Overall_Stars']],
    'star WITHOUT the CAI':      [pc.loc[LA,'rebuilt_no_cai'],pdd.loc[LA,'rebuilt_no_cai'],ov.loc[LA,'rebuilt_no_cai']],
}, index=['Part C', 'Part D', 'Overall']).round(3)
print(la.to_string())

assert (la['rebuilt star'] == la['CMS assigned']).all(), 'L.A. Care does not rebuild'
print('\nAll three of L.A. Care\'s ratings rebuild exactly.')
print('Note the last column: without the CAI, its Part C would round to 2.5, not 3.0.')

wrote contract_cai.csv (769, 7)

Rebuilding the assigned star from the measure stars:

Part C    524 contracts | exact 83.4% | within half a star 100.0% | without CAI 78.1% | CAI moves the star for 42 contracts (8%)
Part D    573 contracts | exact 83.8% | within half a star  99.5% | without CAI 78.9% | CAI moves the star for 37 contracts (6%)
Overall   516 contracts | exact 70.0% | within half a star  98.8% | without CAI 68.0% | CAI moves the star for 62 contracts (12%)

=== L.A. Care (H1224), 2026 ===
         weighted mean of measures  Final Adjustment Category  CAI added  rebuilt star  CMS assigned  star WITHOUT the CAI
Part C                       2.712                        6.0      0.060           3.0           3.0                   2.5
Part D                       3.815                        5.0      0.072           4.0           4.0                   4.0
Overall                      3.089                        8.0      0.101           3.0           3.0                   3.0


## QA - raw gap vs weighted gap: what should L.A. Care actually work on?

The measure gap chart ranks by raw gap, which answers *where is this plan furthest behind*. It does not answer *what should it work on first*, because CMS weights the measures. Multiplying the gap by the weight reorders the list.

In [7]:
nat = (measures[(measures.Year == 2026) & measures.Stars.notna()]
       .groupby('MeasureCode').Stars.mean().rename('national'))

la_gap = (measures[(measures.Contract == LA) & (measures.Year == 2026) & measures.Stars.notna()]
          .merge(weights[['MeasureCode', 'Weight', 'WeightCategory', 'DataSource']], on='MeasureCode')
          .merge(nat, on='MeasureCode'))
la_gap['gap']          = la_gap.Stars - la_gap.national
la_gap['weighted_gap'] = la_gap.gap * la_gap.Weight

cols = ['MeasureName', 'Stars', 'national', 'gap', 'Weight', 'weighted_gap']

print('=== ranked by RAW gap (what the current Tableau chart shows) ===')
print(la_gap.nsmallest(6, 'gap')[cols].round(2).to_string(index=False))

print('\n=== ranked by WEIGHTED gap (where the stars actually are) ===')
print(la_gap.nsmallest(6, 'weighted_gap')[cols].round(2).to_string(index=False))

print('\n=== the reorder ===')
raw_top = la_gap.nsmallest(5, 'gap').MeasureName.tolist()
wtd_top = la_gap.nsmallest(5, 'weighted_gap').MeasureName.tolist()
for i, (r, wgt) in enumerate(zip(raw_top, wtd_top), 1):
    flag = '' if r == wgt else '   <- changed'
    print(f'{i}. raw: {r[:44]:46s} weighted: {wgt[:44]:46s}{flag}')

print('\n=== how much star is recoverable, by data source ===')
print(la_gap.groupby('DataSource')
      .agg(measures=('gap','size'), avg_gap=('gap','mean'), total_weighted_gap=('weighted_gap','sum'))
      .round(2).sort_values('total_weighted_gap').to_string())

=== ranked by RAW gap (what the current Tableau chart shows) ===
                              MeasureName  Stars  national   gap  Weight  weighted_gap
 Medication Reconciliation Post-Discharge    1.0      3.83 -2.83       1         -2.83
                        Care Coordination    1.0      3.50 -2.50       2         -4.99
                         Customer Service    1.0      3.47 -2.47       2         -4.93
Care for Older Adults – Medication Review    2.0      4.16 -2.16       1         -2.16
                      Transitions of Care    1.0      3.11 -2.11       1         -2.11
              Plan All-Cause Readmissions    1.0      2.95 -1.95       3         -5.84

=== ranked by WEIGHTED gap (where the stars actually are) ===
                                   MeasureName  Stars  national   gap  Weight  weighted_gap
                   Plan All-Cause Readmissions    1.0      2.95 -1.95       3         -5.84
                             Care Coordination    1.0      3.50 -2.50       2  